In [2]:
import pandas as pd
import numpy as np
from MDAnalysis import Universe
from MDAnalysis.analysis.distances import distance_array
from calc_df import *
from utils import *
from scipy.spatial import cKDTree
from collections import Counter
import networkx as nx
import os
file_pairs = [
    ('../../resub_traj/n17_k10_II.red.tpr', '../../resub_traj/n17_k10_II.red.xtc'),
    ('../../resub_traj/n17_k10_III.red.tpr', '../../resub_traj/n17_k10_III.red.xtc'),
    ('../../resub_traj/n17_k10_IV.red.tpr', '../../resub_traj/n17_k10_IV.red.xtc')
]

/opt/anaconda3/envs/ML_learn/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
def get_aggregates(grp1, grp2, box):
    # Determine which of the above peptides are aggregates
    pep_aggregates = np.where(distance_array(grp1,grp2,box)<7)
    agg_x = pep_aggregates[0]//pep_len
    agg_y = pep_aggregates[1]//pep_len
    
    # Remove self-interactions
    pairs = [(int(i), int(j)) for i, j in zip(agg_x, agg_y) if i != j]
    
    # Order pairs so that (i,j) is equivalent to (j,i)
    normalized_pairs = [tuple(sorted(pair)) for pair in pairs]  
    
    # Now get counts and create Graph 
    pair_counts = Counter(normalized_pairs)
    interacting_pairs = [ pair for pair, count in pair_counts.items() if count>4]    
    
    G = nx.Graph()
    G.add_nodes_from(range(36))  # assuming 36 peptides always
    G.add_edges_from(interacting_pairs)
    S = [c for c in sorted(nx.connected_components(G), key=len)]

    return S


# Function to get APL values from CSV files
def get_apl_values(frame, file_pair):
    if file_pair == 0:
        filename = f'/Users/nehananajkar/Desktop/Lab/resub_traj/APL_DAT/II_frame{frame}_frame_00000.csv'
        return filename
    elif file_pair == 1:
        filename = f'/Users/nehananajkar/Desktop/Lab/resub_traj/APL_DAT/III_frame{frame}_frame_00000.csv'
        return filename
    elif file_pair == 2:
        filename = f'/Users/nehananajkar/Desktop/Lab/resub_traj/APL_DAT/IV_frame{frame}_frame_00000.csv'
        return filename
    else:
        print('Couldnt find APL file for frame{frame} and traj {file_pair}')
        return None

def get_closest_lipids_resid(peptide_positions, lipid_positions, lipid_resids, k=10):
    """
    Find the k closest lipid atoms (or beads) to a peptide and return their resid values.

    Parameters:
        peptide_positions (np.ndarray): N x 3 array of peptide atom/bead positions.
        lipid_positions (np.ndarray): M x 3 array of lipid atom/bead positions.
        lipid_resids (np.ndarray): M array of lipid residue indices.
        k (int): Number of closest lipids to return.

    Returns:
        closest_positions (np.ndarray): k x 3 array of closest lipid positions.
        closest_resids (np.ndarray): Resid indices of the closest lipids.
    """
    lipid_tree = cKDTree(lipid_positions)
    dists, indices = lipid_tree.query(peptide_positions, k=k)

    if len(peptide_positions) > 1:
        indices = np.unique(indices.flatten())[:k]
    else:
        indices = np.array(indices).flatten()[:k]

    closest_positions = lipid_positions[indices]
    closest_resids = lipid_resids[indices]
    return closest_positions, closest_resids

In [23]:
df = pd.read_csv('skeleton_df.csv')

In [ ]:
apl_data = {}
processed_peptides = set()

for idx, group in df.groupby("trajectory"):
    tpr, xtc = file_pairs[idx]
    u = Universe(tpr, xtc)
    prot = u.select_atoms('name BB S1 S2 S3')
    lip = u.select_atoms('name PO4')
    pep_len = int(len(prot) / 36)
    all_peptides = [prot[i * pep_len : (i + 1) * pep_len] for i in range(36)]
    frames_of_interest = group['frame'].unique()

    for ts in u.trajectory[0:5001]:
        # Skip frames not in the df. 
        if ts.frame not in frames_of_interest:
            continue 
        
        # Get aggregate info for the current frame
        agg_list = get_aggregates(prot, prot, ts.dimensions)
        
        # Get APL data for current frame
        rounded_time = ts.frame // 10
        apl_file = get_apl_values(rounded_time, idx)
        if not os.path.exists(apl_file):
            print(f"File at frame {ts.frame} corresponding to time {rounded_time} is missing")
            continue

        apl_df = pd.read_csv(apl_file)
        
        for current_agg in agg_list:
            agg_status= []
            # Get peptide statuses for the current aggregate
            for current_peptide in current_agg:
                match = group[(group['frame'] == ts.frame) & (group['local_pep_ID'] == current_peptide)]
                if not match.empty:
                    status = match['target_status'].values[0]
                    agg_status.append((current_peptide, status))
            if not agg_status:
                continue  # whole aggregate is in solution — skip
                
            class_01 = [p for p, s in agg_status if s in (0, 1)]
            class_23 = [p for p, s in agg_status if s in (2, 3)]

            if len(class_01) == 0 :
                continue  # current aggregate is not interacting with the membrane
            if len(class_01) == 1:
                core_peptide = class_01[0]
                core_positions = all_peptides[core_peptide].positions
            else:
                core_positions = np.vstack([all_peptides[p].positions for p in class_01])
                
            closest_positions, closest_resids = get_closest_lipids_resid(
            core_positions, lip.positions, lip.resids, k=10)
            mean_apl = apl_df[apl_df['resid'].isin(closest_resids)]['Area per lipid'].mean()
            
            for pep_id, status in agg_status:
                if (ts.frame, pep_id, idx) in processed_peptides:
                    continue
                apl_data[(ts.frame, pep_id, idx)] = mean_apl
                processed_peptides.add((ts.frame, pep_id, idx))





File at frame 83 corresponding to time 8 is missing
File at frame 85 corresponding to time 8 is missing
File at frame 86 corresponding to time 8 is missing
File at frame 87 corresponding to time 8 is missing
File at frame 88 corresponding to time 8 is missing
File at frame 89 corresponding to time 8 is missing
File at frame 90 corresponding to time 9 is missing
File at frame 91 corresponding to time 9 is missing
File at frame 92 corresponding to time 9 is missing
File at frame 93 corresponding to time 9 is missing
File at frame 94 corresponding to time 9 is missing
File at frame 95 corresponding to time 9 is missing
File at frame 96 corresponding to time 9 is missing
File at frame 97 corresponding to time 9 is missing
File at frame 98 corresponding to time 9 is missing
File at frame 99 corresponding to time 9 is missing
File at frame 100 corresponding to time 10 is missing
File at frame 101 corresponding to time 10 is missing
File at frame 102 corresponding to time 10 is missing
File a

In [77]:
apl_df_final = pd.DataFrame(
    [(frame, pep_num, traj, mean_apl) for (frame, pep_num, traj), mean_apl in apl_data.items()],
    columns=['frame', 'local_pep_ID', 'trajectory', 'mean_apl']
)

In [ ]:
# Merge APL info with full group to compare target statuses
merged = pd.merge(
    group,
    apl_df_final,
    on=['frame', 'local_pep_ID', 'trajectory'],
    how='inner'  # Only peptides present in apl_df_final
)

# See what this looks like
print(merged[['frame', 'local_pep_ID', 'trajectory', 'target_status', 'mean_apl']])
len(merged)
# Check for any mismatches or inconsistencies
missing_from_apl = group.merge(apl_df_final, on=['frame', 'local_pep_ID', 'trajectory'], how='left', indicator=True)
not_in_apl = missing_from_apl[missing_from_apl['_merge'] == 'left_only']

#print("\nPeptides in `group` but missing in `apl_df_final`:")
#print(not_in_apl[['frame', 'local_pep_ID', 'trajectory', 'target_status']])


In [78]:
# Check the number of samples for the trajectory ==0 
df = pd.read_csv('skeleton_df.csv')
traj_0 = df.loc[( df['trajectory']==0) & (df['frame']>129),'frame']
len(traj_0)

119916

In [79]:
apl_df_final

,frame,local_pep_ID,trajectory,mean_apl
0,130,10,0,0.5814
1,130,34,0,0.5814
2,130,22,0,0.5814
3,131,10,0,0.5727
4,131,34,0,0.5727
...,...,...,...,...
119911,5000,15,0,0.6495
119912,5000,23,0,0.6495
119913,5000,24,0,0.6495
119914,5000,26,0,0.6495


In [23]:
# Merge APL info with full group to compare target statuses
merged = pd.merge(
    group,
    apl_df_final,
    on=['frame', 'local_pep_ID', 'trajectory'],
    how='inner'  # Only peptides present in apl_df_final
)

# See what this looks like
print(merged[['frame', 'local_pep_ID', 'trajectory', 'target_status', 'mean_apl']])

# Check for any mismatches or inconsistencies
missing_from_apl = group.merge(apl_df_final, on=['frame', 'local_pep_ID', 'trajectory'], how='left', indicator=True)
not_in_apl = missing_from_apl[missing_from_apl['_merge'] == 'left_only']

print("\nPeptides in `group` but missing in `apl_df_final`:")
print(not_in_apl[['frame', 'local_pep_ID', 'trajectory', 'target_status']])


    frame  local_pep_ID  trajectory  target_status  mean_apl
0    5000             3           0              1    0.6495
1    5000             5           0              0    0.9765
2    5000             6           0              0    0.8559
3    5000             9           0              0    0.8559
4    5000            10           0              1    0.6495
5    5000            15           0              1    0.6495
6    5000            17           0              0    1.0754
7    5000            18           0              0    0.8559
8    5000            21           0              0    0.8559
9    5000            22           0              1    0.6085
10   5000            29           0              0    1.0754
11   5000            30           0              0    0.8559
12   5000            31           0              0    0.9765
13   5000            33           0              0    0.8559
14   5000            34           0              1    0.6495

Peptides in `group` but

In [16]:
# Check the number of samples for the trajectory ==0 
df = pd.read_csv('skeleton_df.csv')
traj_0 = df.loc[( df['trajectory']==0) & (df['frame']>129)]
traj_0

,global_pep_ID,local_pep_ID,frame,trajectory,target_status
138,10,10,130,0,3
139,22,22,130,0,1
140,34,34,130,0,3
141,10,10,131,0,3
142,22,22,131,0,1
...,...,...,...,...,...
120049,31,31,5000,0,0
120050,32,32,5000,0,2
120051,33,33,5000,0,0
120052,34,34,5000,0,1


In [6]:
len(df)

221458

In [ ]:
df = pd.read_csv('current_features.csv')
df_merged = pd.merge(df, apl_df_final, on=['frame', 'local_pep_ID', 'trajectory'], how='left')
df_merged.to_csv("./apl_temp.csv")

In [ ]:
################################################################
# Check if the aggregates are correctly identified:
# Peptides classified as 0 should not be part of an aggregate with Class 3 peptides

################################################################
# Check and print target_status for each peptide in aggregates
for i, agg in enumerate(agg_list):
    statuses = []
    for pep_id in agg:
        row = group.loc[(group['frame'] == ts.frame) & (group['local_pep_ID'] == pep_id)]
        if not row.empty:
            status = row['target_status'].values[0]
        else:
            status = 'Not in df'  # likely unclassified (monomer/solution)
        statuses.append((pep_id, status))
    print(f"Aggregate {i}: {statuses}")
    


apl_data = {}
processed_peptides = set()

for idx, group in df.groupby("trajectory"):
    tpr, xtc = file_pairs[0]
    u = Universe(tpr, xtc)
    prot = u.select_atoms('name BB S1 S2 S3')
    lip = u.select_atoms('name PO4')
    pep_len = int(len(prot) / 36)
    all_peptides = [prot[i * pep_len : (i + 1) * pep_len] for i in range(36)]
    frames_of_interest = group['frame'].unique()

    for ts in u.trajectory[4999:5000]:

        if ts.frame in frames_of_interest:
            # Identify peptides directly interacting with bilayer
            filtered = group.loc[
                (group['frame'] == ts.frame) & 
                ((group['target_status'] == 0) | (group['target_status'] == 1))
            ]
            interacting_per_frame = filtered['local_pep_ID'].values
            if len(interacting_per_frame) == 0:
                continue
            print(f"Peptides logged in the df for frame {ts.frame} are : {interacting_per_frame}")
            # Determine distinct peptide aggregates
            agg_list = get_aggregates(prot, prot, ts.dimensions)
            print(f"Current aggregate status: {agg_list}")

            # Determine the rounded time value and open correct APL file
            rounded_time = ts.frame // 10
            apl_file = get_apl_values(rounded_time, idx)
            if not os.path.exists(apl_file):
                print(f'{apl_file} DOES NOT exist')
                continue
            apl_df = pd.read_csv(apl_file)
            
            # Iterate through directly bound peptides (class 0/1)
            for pep_num in interacting_per_frame:
                # Skip if already processed
                if (ts.frame, pep_num, idx) in processed_peptides:
                    continue

                current_peptide = all_peptides[pep_num]
                in_aggregate = [i for i, s in enumerate(agg_list) if pep_num in s]

                if not in_aggregate:
                    # Monomer case
                    print(f"Monomer check - Frame: {ts.frame}, Peptide: {pep_num}, Target Status: \
                    {group[(group['frame'] == ts.frame) & (group['local_pep_ID'] == pep_num)]['target_status'].values}")
                    closest_positions, closest_resids = get_closest_lipids_resid(current_peptide.positions, lip.positions, lip.resids, k=10)
                    mean_apl = apl_df[apl_df['resid'].isin(closest_resids)]['Area per lipid'].mean()
                    apl_data[(ts.frame, pep_num, idx)] = mean_apl
                    processed_peptides.add((ts.frame, pep_num, idx))
                """
                else:
                    agg_idx = in_aggregate[0]
                    print(f"Aggregate detected - Frame: {ts.frame}, Peptide: {pep_num}, Aggregate Members: {agg_list[agg_idx]}")
                    agg_members = agg_list[agg_idx]
                    # Check if current peptide is interacting with other class 0/1 peptides
                    inAgg_touchingMem = [p for p in agg_members if p in interacting_per_frame]

                    # Skip if any touching peptide is already processed
                    if any((ts.frame, p, idx) in processed_peptides for p in inAgg_touchingMem):
                        continue

                    if len(inAgg_touchingMem) == 0:
                        # current peptide is NOT interacting with other class 0/1 peptides
                        closest_positions, closest_resids = get_closest_lipids_resid(current_peptide.positions, lip.positions, lip.resids, k=10)
                        mean_apl = apl_df[apl_df['resid'].isin(closest_resids)]['Area per lipid'].mean()
                        for p in inAgg_touchingMem:
                            if p not in interacting_per_frame:
                                print(f"Potential issue - Frame: {ts.frame}, Aggregate Member: {p} not in interacting_per_frame")
                                apl_data[(ts.frame, p, idx)] = mean_apl
                                processed_peptides.add((ts.frame, p, idx))
                    else:
                        # current peptide IS interacting with other class 0/1 peptides
                        # get apl of all surrounding peps
                        agg_positions = np.vstack([all_peptides[p].positions for p in inAgg_touchingMem])
                        closest_positions, closest_resids = get_closest_lipids_resid(agg_positions, lip.positions, lip.resids, k=10)
                        mean_apl = apl_df[apl_df['resid'].isin(closest_resids)]['Area per lipid'].mean()
                        for p in inAgg_touchingMem:
                            apl_data[(ts.frame, p, idx)] = mean_apl
                            processed_peptides.add((ts.frame, p, idx))
                            
                """
        break

